# 07 – Silver MERGE edge case testleri (P2-A5-S3)

Amaç: Batch log ve quarantine'in edge case'lerde doğru çalıştığını göstermek.

1. Silver reset, batch 1 → 3 runner ile uygulanır.

2. `batch_0002` ikinci kez çalıştırılır: log 4 tabloyu SKIPPED saymalı, silver değişmemeli.

3. Edge case ID'leri silver ve quarantine'de kontrol edilir.

Bu hücre: `src/silver_merge.py` modülünü import eder.

In [0]:
import os
import sys
import importlib

from pyspark.sql import functions as F

# Notebook notebooks/ altında; src klasörü repo kökünde
repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import silver_merge
importlib.reload(silver_merge)

## 1. Reset

Silver 4 tablo bronze snapshot'ına döner (`_change_ts` = 1998-12-31), `card`'da CDF yeniden açılır, `batch_log` ve `quarantine` boşaltılır.

Çıktı nasıl okunur: satır sayıları bronze ile aynı olmalı (client 5.369, account 4.500, card 892, loan 682); `batch_log` ve `quarantine` 0.

In [0]:
silver_merge.reset_silver(spark)

# Reset sonrası başlangıç durumu: silver = bronze, log ve quarantine boş
for table_name in ["client", "account", "card", "loan"]:
    row_count = spark.table(f"workspace.silver.{table_name}").count()
    print(table_name, row_count)

print("batch_log", spark.table("workspace.silver.batch_log").count())
print("quarantine", spark.table("workspace.silver.quarantine").count())

## 2. Batch 1 → 3 (runner)

Her batch'te 4 tablo `run_batch` ile işlenir: log kontrolü, olmayan key'ler quarantine'e, MERGE, log SUCCESS.

Çıktı nasıl okunur: `batch_log`'da 12 satır, hepsi SUCCESS. upd / del / ins sayıları S2 ile aynı olmalı. `rows_skipped` sadece batch 3'te card 1 (101254) ve loan 1 (107308).

In [0]:
CDC_TABLES = ["client", "account", "card", "loan"]

# Sıra önemli: batch'ler birbirinin üstüne kurulu
for batch_id in [1, 2, 3]:
    for table_name in CDC_TABLES:
        silver_merge.run_batch(spark, table_name, batch_id)

df_log = spark.table("workspace.silver.batch_log")
display(
    df_log.select("table_name", "batch_id", "status", "rows_read",
                  "rows_updated", "rows_deleted", "rows_inserted", "rows_skipped")
          .orderBy("batch_id", "table_name")
)

## 3. Tekrardan önceki durum

Her silver tablonun satır sayısı ve son MERGE version'ı kaydedilir. `batch_0002` tekrarından sonra aynı değerler beklenir.

Çıktı nasıl okunur: client 5.383, account 4.500, card 890, loan 682; `batch_log` 12, `quarantine` 2.

In [0]:
def take_snapshot():
    snapshot = {}
    for table_name in CDC_TABLES:
        full_name = f"workspace.silver.{table_name}"
        row_count = spark.table(full_name).count()
        # MERGE çalışırsa yeni version oluşur; OPTIMIZE gibi otomatik işlemler sayılmaz
        df_history = spark.sql(f"DESCRIBE HISTORY {full_name}")
        last_merge_version = df_history.filter(F.col("operation") == "MERGE").agg(F.max("version")).collect()[0][0]
        snapshot[table_name] = {"rows": row_count, "last_merge_version": last_merge_version}
    return snapshot

snapshot_before = take_snapshot()
for table_name in CDC_TABLES:
    print(table_name, snapshot_before[table_name])

log_count_before = spark.table("workspace.silver.batch_log").count()
quarantine_count_before = spark.table("workspace.silver.quarantine").count()
print("batch_log", log_count_before, "| quarantine", quarantine_count_before)

## 4. Aynı batch'in tekrarı (`batch_0002`)

Batch 2 dört tablo için ikinci kez `run_batch`'e verilir. Log'da SUCCESS olduğu için MERGE'e hiç girmemeli.

Çıktı nasıl okunur: her tablo için `run_batch` sonucu SKIPPED, `unchanged` True; `batch_log` 12 → 12, `quarantine` 2 → 2.

In [0]:
# Aynı batch ikinci kez geliyor; log korumasının testi
run_results = {}
for table_name in CDC_TABLES:
    result = silver_merge.run_batch(spark, table_name, 2)
    run_results[table_name] = result

snapshot_after = take_snapshot()

# Satır sayısı ve son MERGE version'ı önce/sonra aynı olmalı
for table_name in CDC_TABLES:
    before = snapshot_before[table_name]
    after = snapshot_after[table_name]
    unchanged = before == after
    print(table_name, "| run_batch:", run_results[table_name], "| before:", before, "| after:", after, "| unchanged:", unchanged)

log_count_after = spark.table("workspace.silver.batch_log").count()
quarantine_count_after = spark.table("workspace.silver.quarantine").count()
print("batch_log", log_count_before, "->", log_count_after)
print("quarantine", quarantine_count_before, "->", quarantine_count_after)

## 5. Edge case kontrolleri

| Edge case | ID | Beklenen |
|---|---|---|
| Aynı key aynı batch'te iki kez (U → U) | client 435, 4206 | Son `change_ts` kazanır: district 28 ve 27 |
| Yeni kart aynı batch'te yükseltildi (I → U) | card 1253 | Son haliyle insert: `gold` |
| Olmayan key'e D / U | card 101254, loan 107308 | Silver'da yok, quarantine'de var |

Çıktı nasıl okunur: `df_checks`'te tüm satırlarda `ok` = true. Altında quarantine'in tamamı (2 satır).

In [0]:
checks = []

# U → U: dedup'ta son change_ts kazanmalı
client_expected = [(435, 28), (4206, 27)]
for client_id, expected_district in client_expected:
    row = spark.table("workspace.silver.client").filter(F.col("client_id") == client_id).collect()[0]
    checks.append((f"client {client_id} district_id", str(expected_district), str(row["district_id"])))

# I → U: kart target'ta yoktu, son haliyle (gold) insert edilmeli
row = spark.table("workspace.silver.card").filter(F.col("card_id") == 1253).collect()[0]
checks.append(("card 1253 type", "gold", row["type"]))

# Olmayan key: silver'a girmemeli, quarantine'de tek satır olmalı
df_quarantine = spark.table("workspace.silver.quarantine")
missing_keys = [("card", "card_id", 101254), ("loan", "loan_id", 107308)]
for table_name, key_column, key_value in missing_keys:
    silver_count = spark.table(f"workspace.silver.{table_name}").filter(F.col(key_column) == key_value).count()
    checks.append((f"{table_name} {key_value} in silver", "0", str(silver_count)))

    quarantine_filter = (F.col("table_name") == table_name) & (F.col("key_value") == str(key_value))
    quarantine_count = df_quarantine.filter(quarantine_filter).count()
    checks.append((f"{table_name} {key_value} in quarantine", "1", str(quarantine_count)))

df_checks = spark.createDataFrame(checks, ["check", "expected", "actual"])
df_checks = df_checks.withColumn("ok", F.col("expected") == F.col("actual"))
display(df_checks)

display(df_quarantine)